<a href="https://colab.research.google.com/github/liuhuhuhuhu/financial-document-intelligence/blob/main/03_embeddings_semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Financial Document Intelligence System

## 03. Embeddings & Semantic Search

**Document:** Apple Inc. FY2025 Form 10-K  
**Tools:** Python, Sentence Transformers, NumPy, Pandas

### Overview

This notebook implements semantic retrieval over financial
document chunks using pretrained sentence embeddings
and cosine similarity.

### Objectives

1. Load structured financial document chunks.
2. Generate 384-dimensional text embeddings.
3. Implement cosine similarity search.
4. Retrieve the top-k relevant chunks for a user query.
5. Preserve document metadata for source attribution.

### Methodology

A pretrained Sentence Transformer model is used to encode
financial disclosures and user queries into a shared
vector space.

Cosine similarity ranks document chunks according to
their semantic relevance to the query.

### Pipeline

Document Chunks → Embeddings → Similarity Search
→ Top-K Retrieved Evidence

**Current stage:** Embedding generation and semantic search.


In [ ]:
!pip -q install sentence-transformers pandas numpy

In [4]:

from google.colab import files

uploaded = files.upload()


Saving apple_2025_chunks.jsonl to apple_2025_chunks.jsonl


In [5]:

import json
import pandas as pd
import numpy as np

with open(
    "apple_2025_chunks.jsonl",
    "r",
    encoding="utf-8"
) as f:
    all_chunks = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

texts = [chunk["text"] for chunk in all_chunks]

print("Total chunks:", len(all_chunks))
print("Example chunk ID:", all_chunks[0]["chunk_id"])
print("Example text:", texts[0][:200])


Total chunks: 368
Example chunk ID: apple_2025_p1_c1
Example text: UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal ye


In [6]:

from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Model loaded:", model_name)
print("Embedding dimensions:",
      model.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimensions: 384


/tmp/ipykernel_849/3074820693.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension())


In [12]:

# Filter table-of-contents chunks before embedding

filtered_chunks = [
    chunk for chunk in all_chunks
    if "TABLE OF CONTENTS" not in chunk["text"].upper()
]

texts = [chunk["text"] for chunk in filtered_chunks]

print("Original chunks:", len(all_chunks))
print("Filtered chunks:", len(filtered_chunks))
print("Removed:", len(all_chunks) - len(filtered_chunks))


Original chunks: 368
Filtered chunks: 367
Removed: 1


In [13]:

chunk_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding matrix shape:",
      chunk_embeddings.shape)


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding matrix shape: (367, 384)


In [14]:

chunk_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", chunk_embeddings.shape)

assert len(filtered_chunks) == len(chunk_embeddings)

print("Embeddings aligned successfully!")


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding shape: (367, 384)
Embeddings aligned successfully!


In [8]:

assert chunk_embeddings.shape == (len(all_chunks), 384)
assert np.isfinite(chunk_embeddings).all()

print("Embedding quality checks passed!")


Embedding quality checks passed!


In [15]:

def semantic_search(query, top_k=5):
    """Retrieve relevant financial disclosure chunks."""

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarity_scores = (
        chunk_embeddings @ query_embedding
    )

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = filtered_chunks[int(idx)]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(similarity_scores[idx]),
            "text": chunk["text"]
        })

    return results


In [18]:

query = "What are Apple's major risk factors?"

results = semantic_search(query, top_k=5)

for result in results:
    print("=" * 70)
    print("Rank:", result["rank"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print("Similarity:", round(result["score"], 4))
    print("Text:")
    print(result["text"][:700])
    print()


Rank: 1
Page: 9
Chunk ID: apple_2025_p9_c8
Similarity: 0.6403
Text:
he Company’s suppliers are required to maintain safe working environments and operations, an industrial accident could occur 
and could result in serious injuries or loss of life, disruption to the Company’s business, and harm to the Company’s reputation. 
Major public health issues, including pandemics such as the COVID-19 pandemic, have adversely affected, and could in the 
future materially adversely affect, the Company due to their impact on the global economy and demand for consumer products; 
the imposition of protective public safety measures, such as stringent employee travel restrictions and limitations on freight 
services and the movement of products between regions; and disruptio

Rank: 2
Page: 11
Chunk ID: apple_2025_p11_c8
Similarity: 0.6204
Text:
In addition, component suppliers may fail, 
be subject to consolidation within a particular industry, or decide to concentrate on the production of common compo

In [17]:

import ipywidgets as widgets
from IPython.display import display, clear_output

search_box = widgets.Text(
    value="What are Apple's major risk factors?",
    placeholder="Enter a financial question...",
    description="Query:",
    layout=widgets.Layout(width="85%")
)

search_button = widgets.Button(
    description="Search Documents",
    button_style="primary"
)

output = widgets.Output()

def on_search_clicked(b):
    with output:
        clear_output()

        query = search_box.value.strip()
        if not query:
            print("Please enter a question.")
            return

        results = semantic_search(query, top_k=5)

        for r in results:
            print(f"Rank {r['rank']} | Page {r['page']}")
            print(f"Similarity: {r['score']:.4f}")
            print(f"Chunk: {r['chunk_id']}")
            print(r["text"][:500])
            print("-" * 65)

search_button.on_click(on_search_clicked)

display(search_box, search_button, output)


Text(value="What are Apple's major risk factors?", description='Query:', layout=Layout(width='85%'), placehold…

Button(button_style='primary', description='Search Documents', style=ButtonStyle())

Output()

feat: implement sentence embeddings and semantic search